# Phase 3 — RL Agent Zoo + Monte Carlo

1. Classical rule agents (`TurtleAgent`, `MovingAverageAgent`, `SignalRollingAgent`, `ABCDStrategyAgent`) on a single ticker.
2. Q-family (`DoubleQAgent`, `DuelQAgent`, `RecurrentQAgent`, `CuriosityQAgent`) on the discrete `StockTradingDiscreteEnv`.
3. Actor-Critic variants and Evolution Strategies on the same env.
4. FinRL ensemble rotator (A2C / PPO / DDPG / TD3 / SAC) on the continuous env.
5. Forward Monte Carlo stress tests — drift / dynamic volatility / multivariate drift.

Deps: `pip install "agentic-quant-platform[ml-torch,streaming]"`.

In [ ]:
import numpy as np
import pandas as pd
from aqp.data.duckdb_engine import DuckDBHistoryProvider

provider = DuckDBHistoryProvider()
bars = provider.get_bars(["SPY"], start="2019-01-01", end="2024-12-31", interval="1d")
bars = bars.set_index("timestamp").sort_index()
bars.tail()

## 1. Classical rule agents

In [ ]:
from aqp.rl.agents.classical import (
    TurtleAgent, MovingAverageAgent, SignalRollingAgent, ABCDStrategyAgent,
)

for cls in (TurtleAgent, MovingAverageAgent, SignalRollingAgent, ABCDStrategyAgent):
    agent = cls(initial_cash=100_000)
    curve = agent.run(bars)
    pv_final = curve['equity'].iloc[-1]
    print(f"{cls.__name__:25s} final equity = {pv_final:,.0f}")

## 2. DQN family on the discrete trading env

Tiny training budget (5 episodes) for illustration — raise for real runs.

In [ ]:
from aqp.rl.envs.stock_trading_discrete import StockTradingDiscreteEnv
from aqp.rl.agents.q_family import (
    QLearningAgent, DoubleQAgent, DuelQAgent, RecurrentQAgent,
)

def _make_env():
    return StockTradingDiscreteEnv(bars.reset_index())

env = _make_env()
obs, _ = env.reset()
state_dim = obs.shape[0] if hasattr(obs, 'shape') else len(obs)

for cls in (QLearningAgent, DoubleQAgent, DuelQAgent):
    agent = cls(state_dim=state_dim, n_actions=env.action_space.n, hidden_size=64)
    returns = agent.train_on_env(env, episodes=3, max_steps=1_000)
    print(f"{cls.__name__:20s} mean episode reward = {np.mean(returns):+.3f}")

## 3. Actor-Critic + Evolution Strategies

Same env, different learning rule.

In [ ]:
from aqp.rl.agents.actor_critic import (
    ActorCriticAgent, ActorCriticDuelAgent, ActorCriticRecurrentAgent,
)

for cls in (ActorCriticAgent, ActorCriticDuelAgent):
    agent = cls(state_dim=state_dim, n_actions=env.action_space.n, hidden_size=64)
    returns = agent.train_on_env(env, episodes=3, max_steps=1_000)
    print(f"{cls.__name__:30s} mean episode reward = {np.mean(returns):+.3f}")

In [ ]:
from aqp.rl.agents.evolutionary import EvolutionStrategyAgent, NeuroEvolutionAgent

es = EvolutionStrategyAgent(state_dim=state_dim, n_actions=env.action_space.n, population=16)
history_es = es.train_on_env(_make_env, generations=5, max_steps=800)
print('ES per-generation mean reward:', history_es)

neo = NeuroEvolutionAgent(state_dim=state_dim, n_actions=env.action_space.n, population=16, elite=3)
history_ne = neo.train_on_env(_make_env, generations=5, max_steps=800)
print('NE per-generation best reward:', history_ne)

## 4. Forward Monte Carlo

Stress-test a policy / strategy against simulated price paths.

In [ ]:
from aqp.backtest.monte_carlo import (
    simulate_drift, simulate_dynamic_volatility,
    simulate_multivariate_drift, portfolio_return_percentiles,
)

drift_paths = simulate_drift(s0=float(bars['close'].iloc[-1]), mu=0.08, sigma=0.20, n_steps=252, n_paths=1_000)
dyn_paths = simulate_dynamic_volatility(
    s0=float(bars['close'].iloc[-1]), mu=0.05, sigma_initial=0.25,
    n_steps=252, n_paths=500, ewma_lambda=0.94, shock_every=63, shock_scale=2.0,
)
print('drift   |', drift_paths.iloc[-1].describe().round(2).to_dict())
print('dyn-vol |', dyn_paths.iloc[-1].describe().round(2).to_dict())

In [ ]:
# Multivariate: grab three correlated tickers and simulate a basket.
tickers = ['SPY', 'QQQ', 'TLT']
prices = (
    provider.get_bars(tickers, start='2021-01-01', end='2024-12-31', interval='1d')
    .pivot(index='timestamp', columns='vt_symbol', values='close').sort_index().ffill()
)
mv = simulate_multivariate_drift(prices, n_steps=126, n_paths=500)
pcts = portfolio_return_percentiles(mv, weights=[0.6, 0.3, 0.1])
pcts